# Minimal AIDA Usage

This notebook shows the smallest public usage path for AIDA.

In [ ]:
# %pip install -e ..

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().resolve().parent.parent
public_src = repo_root / "public" / "src"
if str(public_src) not in sys.path:
    sys.path.insert(0, str(public_src))

from aida import AIDA

In [2]:
csv_path = Path("airline-safety.csv")
if not csv_path.exists():
    df = pd.read_csv("https://raw.githubusercontent.com/fivethirtyeight/data/master/airline-safety/airline-safety.csv")
    df.to_csv(csv_path, index=False)
goal = "Find the most decision-useful patterns in airline safety incidents."

In [3]:
context = AIDA.build_context(csv_path=csv_path, goal=goal, flag_id="airline-safety")
context.row_count, context.schema[:5]

(56,
 [{'column': 'airline', 'dtype': 'str'},
  {'column': 'avail_seat_km_per_week', 'dtype': 'int64'},
  {'column': 'incidents_85_99', 'dtype': 'int64'},
  {'column': 'fatal_accidents_85_99', 'dtype': 'int64'},
  {'column': 'fatalities_85_99', 'dtype': 'int64'}])

In [4]:
# Set one of OPENAI_API_KEY, GEMINI_API_KEY, or DEEPSEEK_API_KEY before running.
 # "models/gemini-flash-lite-latest"#"openai/gpt-5.4-nano" # "deepseek/deepseek-v4-flash"  # openai/gpt-5.4-nano
result = AIDA.run(
    csv_path=csv_path,
    goal=goal,
    model_name="models/gemini-flash-lite-latest", 
    rounds=2,
    questions_per_round=3,
    with_review=False,
)
result.keys()

/Users/ahmed/Aida/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'tuple' object has no attribute 'keys'

In [ ]:
result[0]['final_relevant_insights'][:5]

[{'question_answered': 'Is there a clear trend in safety incident reduction between 1985-1999 and 2000-2014?',
  'insight': 'There has been a significant global improvement in airline safety metrics over the last three decades, with a sharper decline in fatal accidents compared to total incidents.',
  'evidence': 'Total incident volume decreased by 42.5% (from 402 to 231) and total fatal accidents decreased by 69.7% (from 122 to 37) between the two time periods.'},
 {'insight': 'Airline safety performance is highly idiosyncratic, as some carriers have achieved massive improvements while others have experienced significant increases in incident counts over the same period.',
  'evidence': 'Aeroflot saw the largest improvement, reducing its incident count by 70, followed by Ethiopian Airlines with a reduction of 20. Conversely, Southwest Airlines showed the highest increase, adding 7 more incidents in the second period compared to the first.',
  'question_answered': 'Which airlines show 

In [ ]:
result["final_relevant_insights"][:3]

In [ ]:
# Optional post-hoc review of generated insights.
review_payload = AIDA.review(
    goal=goal,
    insights=result["final_relevant_insights"][:3],
    model_name="openai/gpt-4.1-mini",
)
review_payload